# A08_outputs: `households`, the views, and the comparison with prototype 2

| Step | Purpose |
|---|---|
| 8.1 | Final place: the latest-dated current residence in the file |
| 8.2 | Origin: the veteran's enlistment place, else birthplace, else served state |
| 8.3 | Flags: `moved`, `moved_vs_served`, `moved_west` + region |
| 8.4 | `rx-pipeline/schema.sql`: views `moves` and `place_period` |
| 8.5 | Reason codes per household (list), first move range |
| 8.6 | Assertions, save to `processed/`, compare with prototype 2 |

In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote
import pandas as pd

In [2]:
con = remote.connect()
files = io.read_stage("interim", "files")
res = io.read_stage("interim", "residences")
rea = io.read_stage("interim", "reasons")
pl = io.read_stage("processed", "places")
d = io.read_stage("interim", "decl_text", columns=["NAID", "app_year"])
len(files), len(res), len(rea), len(pl)

(78745, 111317, 3206, 1919)

## 8.1 Final place
The `now_resides` row with the latest date (a widow's if hers is later). `declarant` = that row's person.

In [3]:
nr = (res[res.kind == "now_resides"].sort_values(["NAID", "date"], ascending=[True, False], na_position="last")
        .drop_duplicates("NAID"))
h = files[["NAID", "ftype", "claim_nos", "blw_acres", "veteran_name", "served_state"]].rename(columns={"ftype": "claim_type"})
h = h.merge(nr[["NAID", "place_id", "state", "precision", "person"]]
              .rename(columns={"place_id": "final_place_id", "state": "final_state",
                               "precision": "final_precision", "person": "declarant"}), on="NAID", how="left")
h = h.merge(d.groupby("NAID").app_year.max().rename("application_year"), on="NAID", how="left")
print(h.final_precision.value_counts(dropna=False))
h.declarant.value_counts(dropna=False)

final_precision
state     38277
county    26027
NaN       14441
Name: count, dtype: int64


declarant
veteran    35180
widow      19774
<NA>       14441
unknown     9350
Name: count, dtype: int64[pyarrow]

## 8.2 Origin
Veteran `enlisted`, else `born`, else `served_state` (not "Cont").

In [4]:
org = (res[res.kind.isin(["enlisted", "born"]) & (res.person == "veteran")]
         .sort_values(["NAID", "kind"], ascending=[True, False]).drop_duplicates("NAID"))   # 'enlisted' sorts after 'born', so descending puts it first
h = h.merge(org[["NAID", "place_id", "state", "kind"]]
              .rename(columns={"place_id": "origin_place_id", "state": "origin_state", "kind": "origin_kind"}), on="NAID", how="left")
h["origin_state"] = h.origin_state.fillna(h.served_state.where(h.served_state != "Cont"))
h["origin_kind"] = h.origin_kind.fillna(pd.Series("served_state", index=h.index).where(h.origin_state.notna()))
h.origin_kind.value_counts(dropna=False)

origin_kind
served_state    56954
born            10812
NaN              6842
enlisted         4137
Name: count, dtype: int64

## 8.3 Flags
`moved` = origin state ≠ final state · `moved_vs_served` = served ≠ final (the prototype 2 measure) · `moved_west` = origin region east and final region west.

In [5]:
region = pl.set_index("place_id").region
known = h.origin_state.notna() & h.final_state.notna()
h["moved"] = (h.origin_state != h.final_state).where(known).astype("boolean")
srv = h.served_state.notna() & (h.served_state != "Cont") & h.final_state.notna()
h["moved_vs_served"] = (h.served_state != h.final_state).where(srv).astype("boolean")
h["moved_west"] = (h.origin_state.map(region).eq("east") & h.final_state.map(region).eq("west")).where(known).astype("boolean")
print("served vs final:", int(srv.sum()), "both ·", int(h.moved_vs_served.sum()), "moved ·", round(h.moved_vs_served.mean() * 100, 1), "%")
print("origin vs final:", int(known.sum()), "both ·", int(h.moved.sum()), "moved ·", round(h.moved.mean() * 100, 1), "% · moved west", int(h.moved_west.sum()))
h[h.moved_vs_served == True].groupby(["served_state", "final_state"]).size().sort_values(ascending=False).head(8)

served vs final: 60003 both · 32235 moved · 53.7 %
origin vs final: 60477 both · 32838 moved · 54.3 % · moved west 11176


served_state  final_state
CT            NY             3138
MA            ME             2671
              NY             2289
VA            KY             1675
MA            VT             1512
              NH             1121
NC            TN             1041
VA            TN              884
dtype: int64

## 8.4 Views (`rx-pipeline/schema.sql`)
`moves` and `place_period` are *derived*: the rows are stored once as parquet and the derivation lives in SQL, so there is no second copy to drift.

In [6]:
con.register("residences", res); con.register("households", h)
con.sql(open(config.PIPELINE_DIR / "schema.sql").read())
mv = con.sql("select * from moves").df()
print(len(mv), "moves in", mv.NAID.nunique(), "files ·", int((mv.move_after.notna() & mv.move_before.notna()).sum()), "dated at both ends")
early = mv[mv.move_before <= 1810]
print("moves ending by 1810:", len(early), "in", early.NAID.nunique(), "files")
pp = con.sql("select * from place_period order by households desc").df()
print(len(pp), "place_period rows over", pp.place_id.nunique(), "places")
pp.head(8)

18910 moves in 14235 files · 5577 dated at both ends
moves ending by 1810: 1040 in 964 files
3006 place_period rows over 1076 places


,place_id,precision,decade,households,widow_declarants,moved_west
0,NY,state,1830,1790,321,0
1,NY,state,1840,914,678,0
2,NY,state,1820,895,2,0
3,OH,state,1830,688,53,687
4,TN,state,1830,682,38,680
5,KY,state,1830,675,44,671
6,ME,state,1820,649,3,0
7,VT,state,1830,573,146,0


## 8.5 Reasons and first move
With no dated chain: `after = 1783` (end of the war), `before = application_year`, `move_range_method = "service_to_application"`.

In [7]:
h = h.merge(rea.groupby("NAID").reason_code.agg(lambda s: sorted(set(s))).rename("reason_codes"), on="NAID", how="left")
first = mv.sort_values(["NAID", "move_before"]).drop_duplicates("NAID")[["NAID", "move_after", "move_before"]]
h = h.merge(first.rename(columns={"move_after": "first_move_after", "move_before": "first_move_before"}), on="NAID", how="left")
h[["first_move_after", "first_move_before"]] = h[["first_move_after", "first_move_before"]].astype("Int64")
h["move_range_method"] = pd.Series("residence_chain", index=h.index).where(h.first_move_before.notna())
fill = h.moved.fillna(False) & h.first_move_before.isna()
h.loc[fill, "first_move_after"] = 1783
h.loc[fill, "first_move_before"] = h.loc[fill, "application_year"]
h.loc[fill, "move_range_method"] = "service_to_application"
h.move_range_method.value_counts(dropna=False)

move_range_method
NaN                       44790
service_to_application    23600
residence_chain           10355
Name: count, dtype: int64

## 8.6 Assertions + save

In [8]:
assert h.NAID.is_unique and len(h) == len(files)
assert h.final_place_id.dropna().isin(pl.place_id).all()
assert h.origin_place_id.dropna().isin(pl.place_id).all()
assert res.place_id.isin(pl.place_id).all()
io.write_stage(h, "processed", "households")
io.write_stage(res, "processed", "residences")
io.write_stage(rea, "processed", "reasons")
h.dtypes

NAID                     str
claim_type               str
claim_nos             object
blw_acres              Int64
veteran_name             str
served_state          string
final_place_id        string
final_state           string
final_precision          str
declarant             string
application_year       Int64
origin_place_id       string
origin_state          string
origin_kind              str
moved                boolean
moved_vs_served      boolean
moved_west           boolean
reason_codes          object
first_move_after       Int64
first_move_before      Int64
move_range_method        str
dtype: object

## Comparison with prototype 2

| measure | prototype 2 | A08 |
|---|---|---|
| files with served + final state | 59,757 | **60,003** |
| moved (served ≠ final) | 52.8% | **53.7%** (32,235) |
| origin vs. final | n/a | 60,477 both · 54.3% moved · **11,176 moved west** |
| top flows | CT/MA→NY, VA→KY, NC→TN | CT→NY 3,138 · MA→ME 2,671 · MA→NY 2,289 · VA→KY 1,675 · MA→VT 1,512 |
| final place at county level | none | 26,027 files |

**Why it differs (slightly up):**
- every declaration page is read, not only the first, so widow declarations count (`declarant` = widow in 19,774 files)
- abbreviations (`Mass`, `N. Y.`, `Ky`) and 6 more states (AR, WI, IA, FL, TX, DC) resolve
- a county match fixes the state when the text names only the county
- 9,350 files with no stated residence fall back to their most frequent state (`declarant = unknown`); leave them out where the declarant matters

**Caveats:** MA→ME is partly administrative (Maine ⊂ MA until 1820). `moved_west` is a state-level cut, so western NY/PA/VA count as east. Move dates are ranges: 23,600 households get the wide default 1783 → application year; only 10,355 have a dated chain.